In [ ]:
!pip install --user torch==2.8 transformers==4.57.1 datasets==4.0 torchvision==0.23.0

In [ ]:
!pip install rouge

In [ ]:
!pip install -U sentence-transformers

In [ ]:
import torch
from torch.utils.data import DataLoader
import datasets
from transformers import BartForConditionalGeneration, BartTokenizer
from sentence_transformers import SentenceTransformer, util
from rouge import Rouge
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
EPOCHS = 4
BATCH_SIZE = 10
MAX_RESUME_LENGTH = 150
MAX_SUMMARY_LENGTH =  80
LR = 2e-5

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Load model and tokenizer
model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn').to(device)
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')

In [ ]:
# Import evaluation metrics
rouge = Rouge()
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
# Load Data
ds = datasets.load_dataset("burberg92/resume_summary", split="train").shuffle(seed=42)
split_ds = ds.train_test_split(test_size=0.2, seed=42)

# DATA PREPARATION

In [ ]:
def tokenize_dataset(dataset):
    """Process resume dataset to create input_ids, attention_mask, and labels"""
    tokenized_input = []

    for content in dataset:
        inputs = tokenizer(content['resume'], padding='max_length',max_length=MAX_RESUME_LENGTH, truncation=True)
        labels = tokenizer(content['ex_summary'], padding='max_length',max_length=MAX_SUMMARY_LENGTH, truncation=True)
        
        # Replace padding tokens with -100 for BART
        labels = torch.tensor(labels['input_ids'])
        labels[labels == tokenizer.pad_token_id] = -100
        
        tokenized_input.append({'input_ids': torch.tensor(inputs['input_ids']),
                                'attention_mask': torch.tensor(inputs['attention_mask']),
                                'labels': labels})
        
    return tokenized_input

# TRAIN

In [ ]:
def train(dataloader, model, optimizer):
    """Training function for fine-tuning"""
    model.train()
    total_loss = 0

    for batch in dataloader:
        # Move input ids and attention mask to device
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Call model.forward on the items
        preds = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)

        # Compute loss
        loss = preds.loss

        # Zero out the gradients
        optimizer.zero_grad()

        # Call backward on loss
        loss.backward() # find gradient
        optimizer.step() # apply weight updates
        total_loss += loss.item()

    training_loss = total_loss / len(dataloader)
    print("AVERAGE LOSS PER BATCH:", training_loss)
    return training_loss

In [ ]:
def generate_resume(model, tokenizer, resume):
    """Generates a summary for a resume using model"""
    model.eval()

    # tokenize resume
    inputs = tokenizer(resume, return_tensors="pt").to(model.device)

    # generate summary
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=80, min_length=30)

    # decode tokenized summary
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def print_results(scores):
    """Print results as a formatted table"""
    print("="*65)
    print("EVALUATION RESULTS BY RESUME LENGTH")
    print("="*65)
    print(f"{'Length':<10} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<10} {'Cosine Similarity':<10}")
    print("-"*65)
    
    # Print each row except overall
    for size in scores.keys():
        if size != "overall":
            metrics = scores[size]
            print(f"{size:<10} {metrics['rouge-1']:<10.3f} {metrics['rouge-2']:<10.3f} "
                  f"{metrics['rouge-l']:<10.3f} {metrics['cosine']:<10.3f}")
    
    # Print overall
    print("-"*65)
    overall = scores['overall']
    print(f"{'Overall':<10} {overall['rouge-1']:<10.3f} {overall['rouge-2']:<10.3f} "
          f"{overall['rouge-l']:<10.3f} {overall['cosine']:<10.3f}")
    print("="*65)
    print(scores)

In [ ]:
def run_evaluation(dataset, model, tokenizer):
    """Run evaluation using model on dataset. Returns a dictionary containing evaluation
        metrics broken by resume length"""
    lengths = {'100': 0, '200': 0, '300': 0, '400': 0, '500':0,'600':0,'700':0}
    #lengths = {}
    lengths['overall'] = len(dataset)

    scores = {'100': {'rouge-1':0,'rouge-2':0,'rouge-l':0,'cosine':0}, 
          '200': {'rouge-1':0,'rouge-2':0,'rouge-l':0,'cosine':0},
          '300': {'rouge-1':0,'rouge-2':0,'rouge-l':0,'cosine':0}, 
          '400': {'rouge-1':0,'rouge-2':0,'rouge-l':0,'cosine':0}, 
          '500': {'rouge-1':0,'rouge-2':0,'rouge-l':0,'cosine':0},
          '600': {'rouge-1':0,'rouge-2':0,'rouge-l':0,'cosine':0},
          '700': {'rouge-1':0,'rouge-2':0,'rouge-l':0,'cosine':0}, 
          'overall': {'rouge-1':0,'rouge-2':0,'rouge-l':0,'cosine':0}}
    
    #counter = 0
    
    for data in dataset:
        resume = data['resume']
        summary = data['ex_summary']
        
        bart_summary = generate_resume(model, tokenizer, resume)
        
        # # Display first 5 resumes
        # if counter < 5:
        #     print("RESUME: ", resume)
        #     print("GOLD: ", summary)
        #     print("BART: ", bart_summary, "\n")
        #     counter += 1

        rouge_scores = rouge.get_scores(bart_summary, summary)[0]
        
        # Compute embedding for both lists
        embedding_1 = st_model.encode(bart_summary, convert_to_tensor=True)
        embedding_2 = st_model.encode(summary, convert_to_tensor=True)    
        cosine_similarity = util.pytorch_cos_sim(embedding_1, embedding_2).item()
        
        # update overall scores
        scores['overall']['rouge-1'] += rouge_scores['rouge-1']['f']
        scores['overall']['rouge-2'] += rouge_scores['rouge-2']['f']
        scores['overall']['rouge-l'] += rouge_scores['rouge-l']['f']
        scores['overall']['cosine'] += cosine_similarity

        # update length split scores
        group = str(len(resume))[0] + '00'
        #print(group)
        lengths[group] += 1
        scores[group]['rouge-1'] += rouge_scores['rouge-1']['f']
        scores[group]['rouge-2'] += rouge_scores['rouge-2']['f']
        scores[group]['rouge-l'] += rouge_scores['rouge-l']['f']
        scores[group]['cosine'] += cosine_similarity

    for group in scores:
        for metric in scores[group]:
            if lengths[group] != 0:
                scores[group][metric] = scores[group][metric] / lengths[group]

    print_results(scores)
    return scores

# Descriptive Plots

In [ ]:
def plot_performance_by_length(results):
    """Creates a graph of performance by resume length"""
    # Extract data (excluding overall and zero groups)
    lengths = [k for k in results.keys() if results[k]['rouge-1'] > 0]
    metrics = ['rouge-1', 'rouge-2', 'rouge-l', 'cosine']

    # Set up bar positions
    x = np.arange(len(lengths))
    width = 0.2  # Width of bars

    fig, ax = plt.subplots(figsize=(12, 6))

    # Create bars for each metric
    for i, metric in enumerate(metrics):
        values = [results[length][metric] for length in lengths]
        offset = width * (i - 1.5)
        ax.bar(x + offset, values, width, label=metric.upper(), alpha=0.8)

    ax.set_xlabel('Resume Length (characters)', fontsize=14)
    ax.set_ylabel('Score', fontsize=14)
    ax.set_title('Model Performance by Resume Length', fontsize=16)
    ax.set_xticks(x)
    ax.set_xticklabels(lengths)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=5, fontsize=16)
    ax.set_ylim([0, 1.0])
    ax.grid(axis='y', alpha=0.3)

    plt.show()

In [ ]:
def plot_performance(performance):
    """Plot performance of model over epochs"""
    performance = {k: performance[k] for k in performance.keys() if k in ['rouge-1','rouge-2','rouge-l','cosine']}
    
    fig, ax = plt.subplots(figsize=(8,6))
    for k, v in performance.items():
        plt.plot(range(1, len(v) + 1), v, '.-', label=k)
    ax.set_xlabel('Epoch', fontsize=14)
    ax.set_ylabel('Score', fontsize=14)
    ax.set_title('Performance Over Epochs', fontsize=16)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=5, fontsize=16)
    plt.show()

In [ ]:
def plot_training_loss(performance):
    """Plot training loss over epochs"""
    performance = {'training_loss': performance['training_loss']}
    
    fig, ax = plt.subplots(figsize=(8,6))
    for k, v in performance.items():
        plt.plot(range(1, len(v) + 1), v, '.-', label=k)

    ax.set_xlabel('Epoch', fontsize=14)
    ax.set_ylabel('Training Loss', fontsize=14)
    ax.set_title('Training Loss Over Epochs', fontsize=16)
    plt.show()

In [ ]:
def main():
    # Prepare data
    train_dataset = split_ds["train"]
    test_dataset = split_ds["test"]
    train_data = tokenize_dataset(train_dataset)
    test_data = tokenize_dataset(test_dataset)
    train_dataloader= DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=False)
    test_dataloader= DataLoader(test_data, batch_size=1, shuffle=False)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    # Run evaluation on baseline BART model
    scores = run_evaluation(test_dataset, model, tokenizer)
    plot_performance_by_length(scores)

    # Begin finetuning
    print("Finetuning")
    performance = {'training_loss': [], 'rouge-1': [], 'rouge-2': [], 'rouge-l': [], 'cosine': []}
    for e in range(0, EPOCHS):
        print("epoch: ", e)
        training_loss = train(train_dataloader, model, optimizer)
        scores = run_evaluation(test_dataset, model, tokenizer)

        # Plot performance by length
        plot_performance_by_length(scores)

        # Keep track of model performance at this epoch
        performance['training_loss'].append(training_loss)
        performance['rouge-1'].append(scores['overall']['rouge-1'])
        performance['rouge-2'].append(scores['overall']['rouge-2'])
        performance['rouge-l'].append(scores['overall']['rouge-l'])
        performance['cosine'].append(scores['overall']['cosine'])
    
    # Plot model performance
    plot_performance(performance)
    plot_training_loss(performance)


In [ ]:
main()